<a href="https://colab.research.google.com/github/anitejpadyana/BIS-lab/blob/main/Genetic_Algorithm_for_population_based_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

employees = ["Emp1", "Emp2", "Emp3", "Emp4", "Emp5", "Emp6"]
days = ["Day1", "Day2"]
shifts = ["Morning", "Evening", "Night"]

required = 2

unavailable = {
    "Emp1": [],
    "Emp2": ["Night"],
    "Emp3": [],
    "Emp4": ["Night"],
    "Emp5": [],
    "Emp6": []
}

population_size = 100
generations = 500
mutation_rate = 0.1
elite_size = 5

genes = shifts + ["Off"]


def create_chromosome():
    return [
        random.choice(genes)
        for _ in range(len(employees) * len(days))
    ]


def get_assignment(chromosome, employee, day):
    i = employees.index(employee)
    j = days.index(day)
    return chromosome[i * len(days) + j]


def fitness(chromosome):
    penalty = 0
    workload = {e: 0 for e in employees}

    for employee in employees:
        for day in days:
            shift = get_assignment(chromosome, employee, day)

            if shift != "Off":
                workload[employee] += 1

            if shift in unavailable.get(employee, []):
                penalty += 10

    for day in days:
        for shift in shifts:
            count = sum(
                get_assignment(chromosome, employee, day) == shift
                for employee in employees
            )

            if count < required:
                penalty += (required - count) * 20

            if count > required:
                penalty += (count - required) * 20

    max_workload = max(workload.values())
    min_workload = min(workload.values())

    penalty += (max_workload - min_workload) * 5

    return 1 / (1 + penalty)


def selection(population):
    tournament = random.sample(population, 5)
    return max(tournament, key=fitness)


def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)

    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2


def mutate(chromosome):
    chromosome = chromosome.copy()

    for i in range(len(chromosome)):
        if random.random() < mutation_rate:
            chromosome[i] = random.choice(genes)

    return chromosome


population = [
    create_chromosome()
    for _ in range(population_size)
]

for generation in range(generations):

    population.sort(key=fitness, reverse=True)

    new_population = population[:elite_size]

    while len(new_population) < population_size:

        parent1 = selection(population)
        parent2 = selection(population)

        child1, child2 = crossover(parent1, parent2)

        child1 = mutate(child1)
        child2 = mutate(child2)

        new_population.append(child1)

        if len(new_population) < population_size:
            new_population.append(child2)

    population = new_population


best = max(population, key=fitness)

print("Best Fitness:", round(fitness(best), 5))
print()

print("Employee Schedule")
print("-" * 50)

for employee in employees:
    schedule = []

    for day in days:
        schedule.append(get_assignment(best, employee, day))

    print(employee, ":", schedule)


print()
print("Shift Allocation")
print("-" * 50)

for day in days:
    for shift in shifts:

        assigned = [
            employee
            for employee in employees
            if get_assignment(best, employee, day) == shift
        ]

        print(day, shift, ":", assigned)


print()
print("Workload")
print("-" * 50)

for employee in employees:

    workload = sum(
        get_assignment(best, employee, day) != "Off"
        for day in days
    )

    print(employee, ":", workload)


print()
print("Constraint Violations")
print("-" * 50)

violations = 0

for employee in employees:
    for day in days:

        shift = get_assignment(best, employee, day)

        if shift in unavailable.get(employee, []):
            violations += 1

for day in days:
    for shift in shifts:

        count = sum(
            get_assignment(best, employee, day) == shift
            for employee in employees
        )

        violations += abs(required - count)

print("Total Violations:", violations)

Best Fitness: 1.0

Employee Schedule
--------------------------------------------------
Emp1 : ['Evening', 'Morning']
Emp2 : ['Evening', 'Evening']
Emp3 : ['Morning', 'Morning']
Emp4 : ['Morning', 'Evening']
Emp5 : ['Night', 'Night']
Emp6 : ['Night', 'Night']

Shift Allocation
--------------------------------------------------
Day1 Morning : ['Emp3', 'Emp4']
Day1 Evening : ['Emp1', 'Emp2']
Day1 Night : ['Emp5', 'Emp6']
Day2 Morning : ['Emp1', 'Emp3']
Day2 Evening : ['Emp2', 'Emp4']
Day2 Night : ['Emp5', 'Emp6']

Workload
--------------------------------------------------
Emp1 : 2
Emp2 : 2
Emp3 : 2
Emp4 : 2
Emp5 : 2
Emp6 : 2

Constraint Violations
--------------------------------------------------
Total Violations: 0
